# Boréal Marché — Building a Recommendation Engine with Association Rules

**Course:** 420-975-VA — Unsupervised Learning  
**Algorithm:** Apriori (`mlxtend`)  
**Project:** Nermin Koja 

This notebook prepares the transaction data, mines association rules, evaluates the rules, compares seasonal patterns, and exports recommendation rules and metadata for the Streamlit application.

**Important:** Run the notebook from the project root's `notebooks` folder so the relative paths (`../data` and `../models`) work. Run all cells from top to bottom to regenerate the outputs and exported files.


In [1]:
%pip install mlxtend psutil streamlit
import pandas as pd
import numpy as np
import scipy
import mlxtend

print("Python environment is ready!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("mlxtend:", mlxtend.__version__)
import os

print(os.getcwd())
import os

print(os.listdir("../data"))
import pandas as pd

products = pd.read_csv("../data/products.csv")

print(products.head())
print(products.columns.tolist())
print("Number of products:", len(products))
transactions = pd.read_csv(
    "../data/transactions.csv",
    nrows=5
)

print(transactions)
print(transactions.columns.tolist())
transactions = pd.read_csv(
    "../data/transactions.csv",
    usecols=["TransactionID", "Item"]
)

print("Total rows:", len(transactions))
print("Unique baskets:", transactions["TransactionID"].nunique())
print("Unique products:", transactions["Item"].nunique())
print("Average items per basket:",
      len(transactions) / transactions["TransactionID"].nunique())
duplicate_rows = transactions.duplicated(
    subset=["TransactionID", "Item"]
).sum()

print("Duplicate basket-product records:", duplicate_rows)
product_frequency = (
    transactions["Item"]
    .value_counts()
)

print("TOP 10 MOST FREQUENT PRODUCTS:")
print(product_frequency.head(10))

print("\n5 LEAST FREQUENT PRODUCTS:")
print(product_frequency.tail(5))
print(
    products[["Item", "Item_EN", "Category", "Price"]]
    .to_string(index=False)
)



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Python environment is ready!
Pandas: 3.0.6
NumPy: 2.5.3
SciPy: 1.18.1
mlxtend: 0.25.0
c:\Users\Nermin\Desktop\Steve\project\boreal-reco-Nermin\notebooks
['products.csv', 'transactions.csv']
                 Item         Item_EN  Category  Price
0      Steak de boeuf      Beef steak   Viandes  18.99
1           Saucisses        Sausages   Viandes   7.49
2  Poitrine de poulet  Chicken breast   Viandes  12.99
3               Bacon           Bacon   Viandes   6.99
4        Saumon frais    Fresh salmon  Poissons  16.49
['Item', 'Item_EN', 'Category', 'Price']
Number of products: 98
   TransactionID        Date  CustomerID                 Item
0         100000  2026-06-19       16595       Cidre de glace
1         100000  2026-06-19       16595              Couches
2         100000  2026-06-19       16595      Fromage cheddar
3         100000  2026-06-19       16595  Lingettes pour bébé
4         100000  2026-06-19       16595

### Catalogue duplicate investigation

The catalogue contains two similar products:

- Baguette — $2.99
- Pain baguette rustique (Rustic baguette) — $3.99

Although both products belong to the baguette family, they are not necessarily the same product. They have different names and prices, and the rustic version may represent a different product.

**Decision: Keep both products separate.**

Merging them could remove useful information about customer purchasing preferences and product differences.


In [2]:
print(
    "Transaction data memory:",
    round(transactions.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)
row_codes = transactions["TransactionID"].astype("category")
item_codes = transactions["Item"].astype("category")

print("Number of baskets:", len(row_codes.cat.categories))
print("Number of products:", len(item_codes.cat.categories))
from scipy import sparse
import numpy as np
import time

start_time = time.perf_counter()

basket_matrix = sparse.csr_matrix(
    (
        np.ones(len(transactions), dtype=bool),
        (row_codes.cat.codes, item_codes.cat.codes),
    ),
    shape=(len(row_codes.cat.categories), len(item_codes.cat.categories)),
)

elapsed_time = time.perf_counter() - start_time

print("Basket matrix created!")
print("Shape:", basket_matrix.shape)
print("Time:", round(elapsed_time, 2), "seconds")
matrix_memory = (
    basket_matrix.data.nbytes
    + basket_matrix.indices.nbytes
    + basket_matrix.indptr.nbytes
)

print("Sparse matrix memory:",
      round(matrix_memory / 1024**2, 2), "MB")
print("First 3 baskets and first 5 products:")
print(basket_matrix[:3, :5].toarray())
basket_df = pd.DataFrame.sparse.from_spmatrix(
    basket_matrix,
    index=row_codes.cat.categories,
    columns=item_codes.cat.categories
)

print("Basket table shape:", basket_df.shape)
print(basket_df.iloc[:3, :5])
print("Purchases in original data:", len(transactions))
print("Purchases in basket matrix:", basket_matrix.nnz)
basket_df_memory = (
    basket_df.memory_usage(index=True, deep=True).sum()
)

print(
    "Pandas sparse basket table memory:",
    round(basket_df_memory / 1024**2, 2),
    "MB"
)
basket_frequency = basket_df.sum(axis=0).sort_values(
    ascending=False
)

print("Top 10 products:")
print(basket_frequency.head(10))
basket_frequency = (
    basket_df.astype(int)
    .sum(axis=0)
    .sort_values(ascending=False)
)

print("Top 10 products:")
print(basket_frequency.head(10))
print("Five rarest products:")
print(basket_frequency.tail(5))
product_support = basket_frequency / len(basket_df)

print("Top 10 products by support:")
print(product_support.sort_values(ascending=False).head(10))
print("Five rarest products by support:")
print(product_support.sort_values().head(5))


Transaction data memory: 230.42 MB
Number of baskets: 1000000
Number of products: 98
Basket matrix created!
Shape: (1000000, 98)
Time: 0.18 seconds
Sparse matrix memory: 43.77 MB
First 3 baskets and first 5 products:
[[False False False False False]
 [False False False False False]
 [ True False False False False]]
Basket table shape: (1000000, 98)
          Ail  Algues nori  Avocat  Bacon  Bagels
100000  False        False   False  False   False
100001  False        False   False  False   False
100002   True        False   False  False   False
Purchases in original data: 8379837
Purchases in basket matrix: 8379837
Pandas sparse basket table memory: 47.59 MB
Top 10 products:
Ail                     True
Riz basmati             True
Pâtes alimentaires      True
Purée pour bébé         True
Pommes de terre         True
Pommes                  True
Poivron rouge           True
Poitrine de poulet      True
Pizza surgelée          True
Petits pois surgelés    True
dtype: Sparse[bool, False]

### Product Frequency and Support Analysis

The product frequency analysis shows that Sac réutilisable is the most frequently purchased product, appearing in 900,026 baskets (90.0026% support).

The five rarest products are Foie gras, Vinaigre balsamique vieilli, Caviar, Huile de truffe, and Safran.

Foie gras appears in only 756 baskets, representing 0.0756% support.

The lowest Apriori threshold required by the project is 0.005 (0.5%), or 5,000 baskets. Therefore, these five rare products cannot appear in frequent itemsets at any of the four required thresholds.

This may limit recommendations for customers purchasing rare products.


In [3]:
print("Basket table shape:", basket_df.shape)
print("Data types:")
print(basket_df.dtypes.value_counts())
print("Sparse table:", all(
    isinstance(dtype, pd.SparseDtype)
    for dtype in basket_df.dtypes
))


Basket table shape: (1000000, 98)
Data types:
Sparse[bool, False]    98
Name: count, dtype: int64
Sparse table: True


### Initial Prediction Before Apriori Mining

Before applying Apriori, I predict that **Sac réutilisable** may be a troublesome product when evaluating association rules.

It appears in approximately 90% of all baskets. Therefore, rules involving this product may have high confidence simply because it is already extremely popular, rather than because it has a strong association with another product.

I will compare confidence and lift after mining to test this prediction.


In [4]:
min_support = 0.05

print("Apriori experiment")
print("Minimum support:", min_support)
print("Minimum baskets required:",
      int(min_support * len(basket_df)))
print("Number of baskets:", len(basket_df))
print("Number of products:", len(basket_df.columns))
from mlxtend.frequent_patterns import apriori

start_time = time.perf_counter()

frequent_itemsets = apriori(
    basket_df,
    min_support=0.05,
    use_colnames=True,
    low_memory=True
)

apriori_time = time.perf_counter() - start_time

print("Apriori completed!")
print("Number of frequent itemsets:", len(frequent_itemsets))
print("Execution time:", round(apriori_time, 2), "seconds")
print(frequent_itemsets.head(15))
frequent_itemsets["itemset_size"] = (
    frequent_itemsets["itemsets"].apply(len)
)

print("Itemsets by size:")
print(frequent_itemsets["itemset_size"].value_counts().sort_index())
print(
    frequent_itemsets.sort_values(
        by="support",
        ascending=False
    ).head(10)
)
largest_itemsets = frequent_itemsets[
    frequent_itemsets["itemset_size"] == 4
]

print(largest_itemsets)
for _, row in largest_itemsets.iterrows():
    print("Support:", round(row["support"], 6))
    print("Products:", sorted(row["itemsets"]))
    print("-" * 50)
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.0
)

print("Number of association rules:", len(rules))
print(rules.head())


Apriori experiment
Minimum support: 0.05
Minimum baskets required: 50000
Number of baskets: 1000000
Number of products: 98
Apriori completed!
Number of frequent itemsets: 191
Execution time: 7.48 seconds
     support                        itemsets
0   0.084526                frozenset({Ail})
1   0.068465        frozenset({Algues nori})
2   0.085813              frozenset({Bacon})
3   0.062716             frozenset({Bagels})
4   0.069400           frozenset({Baguette})
5   0.134223             frozenset({Beurre})
6   0.153726  frozenset({Beurre d'arachide})
7   0.161439           frozenset({Biscuits})
8   0.150468              frozenset({Bière})
9   0.125887         frozenset({Café moulu})
10  0.056148        frozenset({Champignons})
11  0.140561    frozenset({Charbon de bois})
12  0.072676      frozenset({Chocolat noir})
13  0.058224     frozenset({Cidre de glace})
14  0.069634          frozenset({Concombre})
Itemsets by size:
itemset_size
1    72
2    92
3    24
4     3
Name: count, 

### Understanding Support, Confidence, and Lift

- **Support** tells us how often the products in a rule appear together in all transactions. A higher support means the combination occurs more frequently in the dataset.
- **Confidence** tells us how often the consequent product is present in baskets that already contain the antecedent product or products. It measures how reliable the rule is within those baskets.
- **Lift** compares the rule's confidence with how common the recommended product is overall. A lift greater than 1 means the products occur together more often than expected if they were independent. A lift close to 1 indicates a weak association, even if confidence is high.

These three measurements answer different questions, so I evaluate them together rather than relying on confidence alone.


In [5]:
top_lift_rules = rules.sort_values(
    by="lift",
    ascending=False
)

print(
    top_lift_rules[
        ["antecedents", "consequents", "support",
         "confidence", "lift"]
    ].head(10)
)
simple_rules = rules[
    (rules["antecedents"].apply(len) == 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

print("Number of simple rules:", len(simple_rules))
print("Top 10 simple rules by lift:")

print(
    simple_rules.sort_values(
        "lift", ascending=False
    )[
        ["antecedents", "consequents",
         "support", "confidence", "lift"]
    ].head(10)
)
print("Top 10 simple rules by confidence:")

print(
    simple_rules.sort_values(
        "confidence", ascending=False
    )[
        ["antecedents", "consequents",
         "support", "confidence", "lift"]
    ].head(10)
)


                                           antecedents  \
6                             frozenset({Algues nori})   
7                             frozenset({Riz à sushi})   
58                    frozenset({Lingettes pour bébé})   
59                                frozenset({Couches})   
275  frozenset({Lingettes pour bébé, Sac réutilisab...   
278                               frozenset({Couches})   
277                   frozenset({Lingettes pour bébé})   
276             frozenset({Couches, Sac réutilisable})   
77                       frozenset({Frites surgelées})   
76                      frozenset({Fromage en grains})   

                                           consequents   support  confidence  \
6                             frozenset({Riz à sushi})  0.054431    0.795019   
7                             frozenset({Algues nori})  0.054431    0.650738   
58                                frozenset({Couches})  0.055900    0.796762   
59                    frozenset({Lingette

### Comparing Confidence and Lift Rankings

The top 10 simple rules ranked by confidence all recommend Sac réutilisable.

Their confidence values are approximately 90%, but their lift values are very close to 1.

This happens because Sac réutilisable already appears in 90.0026% of all baskets. Therefore, its high confidence does not indicate a meaningful purchasing relationship.

In contrast, the top rules ranked by lift reveal stronger associations, such as Algues nori → Riz à sushi, with a lift of approximately 9.50.

This confirms our initial prediction that Sac réutilisable could be a troublesome product. It also demonstrates why confidence and lift should be considered together when evaluating recommendations.


In [19]:

import psutil
import tracemalloc
import threading
import time
import pandas as pd

print("Memory measurement tools are ready!")

process = psutil.Process()


def run_apriori_benchmark(label, min_support, low_memory=False):
    """Run Apriori and measure execution time, RSS, and traced memory."""

    peak_memory = 0
    stop_monitoring = threading.Event()

    def monitor_memory():
        nonlocal peak_memory

        while not stop_monitoring.is_set():
            current_memory = process.memory_info().rss / (1024 ** 2)
            peak_memory = max(peak_memory, current_memory)
            time.sleep(0.05)

    tracemalloc.start()
    monitor = threading.Thread(target=monitor_memory)
    monitor.start()

    start_time = time.perf_counter()

    try:
        itemsets = apriori(
            basket_df,
            min_support=min_support,
            use_colnames=True,
            low_memory=low_memory
        )
    finally:
        execution_time = time.perf_counter() - start_time
        stop_monitoring.set()
        monitor.join()

        current_memory, peak_traced_memory = (
            tracemalloc.get_traced_memory()
        )
        tracemalloc.stop()

    peak_traced_memory_mb = peak_traced_memory / (1024 ** 2)

    print("\n" + label)
    print("Frequent itemsets:", len(itemsets))
    print("Execution time:", round(execution_time, 2), "seconds")
    print("Peak RSS:", round(peak_memory, 2), "MB")
    print("Peak traced memory:", round(peak_traced_memory_mb, 2), "MB")

    return {
        "itemsets": itemsets,
        "execution_time": execution_time,
        "peak_rss": peak_memory,
        "peak_traced_memory": peak_traced_memory_mb
    }


# Default Apriori: support 0.05
default_result = run_apriori_benchmark(
    "Default Apriori — support 0.05",
    min_support=0.05
)
frequent_itemsets_default = default_result["itemsets"]
apriori_default_time = default_result["execution_time"]


# Low-memory Apriori: support 0.05
low_memory_result = run_apriori_benchmark(
    "Low-memory Apriori — support 0.05",
    min_support=0.05,
    low_memory=True
)
frequent_itemsets_low_memory = low_memory_result["itemsets"]
low_memory_time = low_memory_result["execution_time"]


# Low-memory Apriori: support 0.02
result_002 = run_apriori_benchmark(
    "Low-memory Apriori — support 0.02",
    min_support=0.02,
    low_memory=True
)
frequent_itemsets_002 = result_002["itemsets"]
time_002 = result_002["execution_time"]


# Low-memory Apriori: support 0.01
result_001 = run_apriori_benchmark(
    "Low-memory Apriori — support 0.01",
    min_support=0.01,
    low_memory=True
)
frequent_itemsets_001 = result_001["itemsets"]
time_001 = result_001["execution_time"]


# Low-memory Apriori: support 0.005
result_0005 = run_apriori_benchmark(
    "Low-memory Apriori — support 0.005",
    min_support=0.005,
    low_memory=True
)
frequent_itemsets_0005 = result_0005["itemsets"]
time_0005 = result_0005["execution_time"]


# Count association rules using existing frequent itemsets
benchmark_itemsets = {
    "Default 0.05": frequent_itemsets_default,
    "Low memory 0.05": frequent_itemsets_low_memory,
    "Low memory 0.02": frequent_itemsets_002,
    "Low memory 0.01": frequent_itemsets_001,
    "Low memory 0.005": frequent_itemsets_0005
}

benchmark_rule_counts = {}

for label, itemsets in benchmark_itemsets.items():
    generated_rules = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=0.0
    )
    benchmark_rule_counts[label] = len(generated_rules)


# Create benchmark table using actual measurements
benchmark_runs = [
    default_result,
    low_memory_result,
    result_002,
    result_001,
    result_0005
]

benchmark_results = pd.DataFrame({
    "Apriori mode": [
        "Default",
        "Low memory",
        "Low memory",
        "Low memory",
        "Low memory"
    ],
    "Support": [0.05, 0.05, 0.02, 0.01, 0.005],
    "Frequent itemsets": [
        len(run["itemsets"]) for run in benchmark_runs
    ],
    "Association rules": [
        benchmark_rule_counts[label]
        for label in benchmark_itemsets
    ],
    "Execution time (seconds)": [
        round(run["execution_time"], 2)
        for run in benchmark_runs
    ],
    "Sampled peak RSS (MB)": [
        round(run["peak_rss"], 2)
        for run in benchmark_runs
    ],
    "Peak traced memory (MB)": [
        round(run["peak_traced_memory"], 2)
        for run in benchmark_runs
    ]
})

benchmark_results


# Seasonal analysis
transactions_dates = pd.read_csv(
    "../data/transactions.csv",
    usecols=["Date"],
    parse_dates=["Date"]
)

print("Earliest date:", transactions_dates["Date"].min())
print("Latest date:", transactions_dates["Date"].max())

transactions_dates.head()

transactions_dates["Month"] = transactions_dates["Date"].dt.month

monthly_counts = transactions_dates["Month"].value_counts().sort_index()

monthly_counts

transactions_dates["Season"] = transactions_dates["Month"].map({
    6: "Summer",
    7: "Summer",
    8: "Summer",
    12: "Winter",
    1: "Winter",
    2: "Winter"
})

transactions_dates["Season"].value_counts()

transactions_with_dates = pd.read_csv(
    "../data/transactions.csv",
    usecols=["TransactionID", "Date", "Item"],
    parse_dates=["Date"]
)

transactions_with_dates["Month"] = (
    transactions_with_dates["Date"].dt.month
)

transactions_with_dates["Season"] = (
    transactions_with_dates["Month"].map({
        6: "Summer",
        7: "Summer",
        8: "Summer",
        12: "Winter",
        1: "Winter",
        2: "Winter"
    })
)

season_baskets = (
    transactions_with_dates
    .dropna(subset=["Season"])
    .groupby("Season")["TransactionID"]
    .nunique()
)

season_baskets

summer_transactions = transactions_with_dates[
    transactions_with_dates["Season"] == "Summer"
]

summer_basket_df = (
    summer_transactions
    .assign(value=True)
    .pivot_table(
        index="TransactionID",
        columns="Item",
        values="value",
        aggfunc="any",
        fill_value=False
    )
)

print("Summer basket table shape:", summer_basket_df.shape)

winter_transactions = transactions_with_dates[
    transactions_with_dates["Season"] == "Winter"
]

winter_basket_df = (
    winter_transactions
    .assign(value=True)
    .pivot_table(
        index="TransactionID",
        columns="Item",
        values="value",
        aggfunc="any",
        fill_value=False
    )
)

print("Winter basket table shape:", winter_basket_df.shape)

summer_itemsets = apriori(
    summer_basket_df,
    min_support=0.02,
    use_colnames=True,
    low_memory=True
)

print("Summer frequent itemsets:", len(summer_itemsets))

winter_itemsets = apriori(
    winter_basket_df,
    min_support=0.02,
    use_colnames=True,
    low_memory=True
)

print("Winter frequent itemsets:", len(winter_itemsets))

summer_rules = association_rules(
    summer_itemsets,
    metric="confidence",
    min_threshold=0.0
)

print("Summer association rules:", len(summer_rules))

winter_rules = association_rules(
    winter_itemsets,
    metric="confidence",
    min_threshold=0.0
)

print("Winter association rules:", len(winter_rules))

summer_simple_rules = summer_rules[
    (summer_rules["antecedents"].apply(len) == 1) &
    (summer_rules["consequents"].apply(len) == 1)
].copy()

winter_simple_rules = winter_rules[
    (winter_rules["antecedents"].apply(len) == 1) &
    (winter_rules["consequents"].apply(len) == 1)
].copy()

print("Summer simple rules:", len(summer_simple_rules))
print("Winter simple rules:", len(winter_simple_rules))

summer_rule_pairs = set(
    zip(
        summer_simple_rules["antecedents"].apply(
            lambda x: next(iter(x))
        ),
        summer_simple_rules["consequents"].apply(
            lambda x: next(iter(x))
        )
    )
)

winter_rule_pairs = set(
    zip(
        winter_simple_rules["antecedents"].apply(
            lambda x: next(iter(x))
        ),
        winter_simple_rules["consequents"].apply(
            lambda x: next(iter(x))
        )
    )
)

summer_only_pairs = summer_rule_pairs - winter_rule_pairs

print("Summer-only rules:", len(summer_only_pairs))

summer_only_examples = summer_simple_rules[
    summer_simple_rules.apply(
        lambda row: (
            next(iter(row["antecedents"])),
            next(iter(row["consequents"]))
        ) in summer_only_pairs,
        axis=1
    )
].sort_values("lift", ascending=False)

summer_only_examples[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(10)

winter_only_pairs = winter_rule_pairs - summer_rule_pairs

print("Winter-only rules:", len(winter_only_pairs))

winter_only_examples = winter_simple_rules[
    winter_simple_rules.apply(
        lambda row: (
            next(iter(row["antecedents"])),
            next(iter(row["consequents"]))
        ) in winter_only_pairs,
        axis=1
    )
].sort_values("lift", ascending=False)

winter_only_examples[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(10)


Memory measurement tools are ready!

Default Apriori — support 0.05
Frequent itemsets: 191
Execution time: 38.89 seconds
Peak RSS: 7300.61 MB
Peak traced memory: 8689.47 MB

Low-memory Apriori — support 0.05
Frequent itemsets: 191
Execution time: 3.98 seconds
Peak RSS: 263.26 MB
Peak traced memory: 217.93 MB

Low-memory Apriori — support 0.02
Frequent itemsets: 658
Execution time: 7.49 seconds
Peak RSS: 322.65 MB
Peak traced memory: 262.34 MB

Low-memory Apriori — support 0.01
Frequent itemsets: 2267
Execution time: 18.58 seconds
Peak RSS: 321.7 MB
Peak traced memory: 262.31 MB

Low-memory Apriori — support 0.005
Frequent itemsets: 7106
Execution time: 51.72 seconds
Peak RSS: 323.58 MB
Peak traced memory: 262.32 MB
Earliest date: 2026-01-01 00:00:00
Latest date: 2026-12-31 00:00:00
Summer basket table shape: (252474, 98)
Winter basket table shape: (246965, 98)
Summer frequent itemsets: 985
Winter frequent itemsets: 564
Summer association rules: 8904
Winter association rules: 2382
Summe

,antecedents,consequents,support,confidence,lift
135,frozenset({Bouillon à fondue}),frozenset({Fromage à raclette}),0.067248,0.829653,6.808285
134,frozenset({Fromage à raclette}),frozenset({Bouillon à fondue}),0.067248,0.551852,6.808285
182,frozenset({Fromage à raclette}),frozenset({Charcuteries}),0.064025,0.525403,6.718242
183,frozenset({Charcuteries}),frozenset({Fromage à raclette}),0.064025,0.818681,6.718242
211,frozenset({Cornichons}),frozenset({Fromage à raclette}),0.059199,0.813941,6.679343
210,frozenset({Fromage à raclette}),frozenset({Cornichons}),0.059199,0.485795,6.679343
307,frozenset({Pain baguette rustique}),frozenset({Fromage à raclette}),0.055162,0.796713,6.537973
306,frozenset({Fromage à raclette}),frozenset({Pain baguette rustique}),0.055162,0.452667,6.537973
313,frozenset({Fromage à raclette}),frozenset({Vin blanc}),0.050667,0.415783,6.456079
312,frozenset({Vin blanc}),frozenset({Fromage à raclette}),0.050667,0.786734,6.456079


In [21]:
benchmark_results

,Apriori mode,Support,Frequent itemsets,Association rules,Execution time (seconds),Sampled peak RSS (MB),Peak traced memory (MB)
0,Default,0.050,191,370,38.89,7300.61,8689.47
1,Low memory,0.050,191,370,3.98,263.26,217.93
2,Low memory,0.020,658,3746,7.49,322.65,262.34
3,Low memory,0.010,2267,16096,18.58,321.70,262.31
4,Low memory,0.005,7106,63662,51.72,323.58,262.32


In [17]:
# Count association rules from the already-generated frequent itemsets.
# This does NOT rerun Apriori.

benchmark_rule_counts = {}

for label, itemsets in benchmark_itemsets.items():
    generated_rules = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=0.0
    )

    benchmark_rule_counts[label] = len(generated_rules)

print("Benchmark association-rule counts:")

for label, count in benchmark_rule_counts.items():
    print(f"{label}: {count}")

Benchmark association-rule counts:
Default 0.05: 370
Low memory 0.05: 370
Low memory 0.02: 3746
Low memory 0.01: 16096
Low memory 0.005: 63662


### Comparing the Same Rules Across Summer and Winter

The next cell matches rules that appear in both seasonal datasets and compares their support, confidence, and lift side by side. This is important because comparing only the number of rules or separate seasonal examples does not show how the strength of the same association changes between seasons.


In [7]:
def prepare_seasonal_metrics(season_rules):
    comparison = season_rules.copy()
    comparison["antecedent_key"] = comparison["antecedents"].apply(
        lambda items: tuple(sorted(items))
    )
    comparison["consequent_key"] = comparison["consequents"].apply(
        lambda items: tuple(sorted(items))
    )
    return comparison[
        ["antecedent_key", "consequent_key", "support", "confidence", "lift"]
    ]

summer_metrics = prepare_seasonal_metrics(summer_simple_rules)
winter_metrics = prepare_seasonal_metrics(winter_simple_rules)

shared_seasonal_comparison = summer_metrics.merge(
    winter_metrics,
    on=["antecedent_key", "consequent_key"],
    how="inner",
    suffixes=("_summer", "_winter")
)

shared_seasonal_comparison["lift_difference"] = (
    shared_seasonal_comparison["lift_summer"]
    - shared_seasonal_comparison["lift_winter"]
).abs()

print("Rules present in both seasons:", len(shared_seasonal_comparison))
display(
    shared_seasonal_comparison.sort_values(
        "lift_difference", ascending=False
    )[
        [
            "antecedent_key", "consequent_key",
            "support_summer", "support_winter",
            "confidence_summer", "confidence_winter",
            "lift_summer", "lift_winter"
        ]
    ]
)

print("Interpretation:")
print(
    "The table compares the same directional product rules in both seasons. "
    "Differences in support, confidence, and lift show whether an association "
    "becomes more or less common or reliable in one season."
)


Rules present in both seasons: 372


,antecedent_key,consequent_key,support_summer,support_winter,confidence_summer,confidence_winter,lift_summer,lift_winter
142,"(Charbon de bois,)","(Steak de boeuf,)",0.187334,0.025145,0.694808,0.509016,3.498625,12.660815
143,"(Steak de boeuf,)","(Charbon de bois,)",0.187334,0.025145,0.943299,0.625441,3.498625,12.660815
139,"(Pain à hamburger,)","(Charbon de bois,)",0.177187,0.023550,0.940404,0.605960,3.487887,12.266460
138,"(Charbon de bois,)","(Pain à hamburger,)",0.177187,0.023550,0.657172,0.476721,3.487887,12.266460
136,"(Charbon de bois,)","(Ketchup,)",0.156903,0.020971,0.581943,0.424508,3.455036,11.906720
...,...,...,...,...,...,...,...,...
126,"(Carottes,)","(Sac réutilisable,)",0.028336,0.027631,0.898292,0.897659,0.997451,0.997342
240,"(Jus d'orange,)","(Sac réutilisable,)",0.059404,0.059207,0.899700,0.899207,0.999014,0.999062
241,"(Sac réutilisable,)","(Jus d'orange,)",0.059404,0.059207,0.065962,0.065782,0.999014,0.999062
56,"(Sac réutilisable,)","(Beurre,)",0.121264,0.119835,0.134650,0.133142,0.999664,0.999711


Interpretation:
The table compares the same directional product rules in both seasons. Differences in support, confidence, and lift show whether an association becomes more or less common or reliable in one season.


## Seasonal Analysis

The dataset was divided into two seasonal groups:

- Summer: June, July, and August.
- Winter: December, January, and February.

The same minimum support threshold (0.02) was used for both seasons.

### Results

| Measurement | Summer | Winter |
|---|---:|---:|
| Baskets | 252,474 | 246,965 |
| Frequent itemsets | 985 | 564 |
| Association rules | 8,904 | 2,382 |
| Simple rules | 596 | 478 |

There were 224 rules found only in summer and 106 rules found only in winter.

### Summer example

Charcoal → Sausages

- Support: 14.38%
- Confidence: 53.32%
- Lift: 3.44

This association may reflect summer barbecue purchasing patterns.

### Winter example

Raclette cheese → Fondue broth

- Support: 6.72%
- Confidence: 55.19%
- Lift: 6.81

This association may reflect winter raclette and fondue meals.

### Interpretation

The results show that purchasing associations can vary by season. A recommendation engine using only year-round rules may miss useful seasonal relationships.

A rule absent from one seasonal rule set did not necessarily have zero purchases together; it simply did not meet the selected support threshold of 0.02.

### Limitation

The dataset covers only 2026. Winter combines January–February 2026 with December 2026. More years of data would help validate whether these patterns repeat each year.
### Recommended Re-mining Frequency

I recommend re-mining the association rules every month.

Monthly updates allow the recommendation engine to adapt to changing customer purchasing habits and emerging product associations.

Seasonal analysis should also be used to identify patterns that may be specific to summer or winter.

Using only year-round rules could hide seasonal relationships and lead to less relevant recommendations, especially when winter purchasing habits differ from summer habits.


### Selecting Rules for the Recommendation Engine

The professor's requirement is to retain rules with **lift greater than 1**. For a product recommendation, the consequent must contain exactly one product so that the application can display one recommended product at a time.

The antecedent is allowed to contain one or more products. Keeping multi-item antecedents is necessary for recommendations triggered by combinations of products in the shopping basket.


In [8]:
selected_rules = rules[
    (rules["lift"] > 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

print("Selected recommendation rules:", len(selected_rules))
print("Rules with multi-product antecedents:",
      (selected_rules["antecedents"].apply(len) > 1).sum())

selected_rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].sort_values("lift", ascending=False).head(10)


Selected recommendation rules: 215
Rules with multi-product antecedents: 81


,antecedents,consequents,support,confidence,lift
6,frozenset({Algues nori}),frozenset({Riz à sushi}),0.054431,0.795019,9.504685
7,frozenset({Riz à sushi}),frozenset({Algues nori}),0.054431,0.650738,9.504685
58,frozenset({Lingettes pour bébé}),frozenset({Couches}),0.055900,0.796762,9.489098
59,frozenset({Couches}),frozenset({Lingettes pour bébé}),0.055900,0.665746,9.489098
275,"frozenset({Lingettes pour bébé, Sac réutilisab...",frozenset({Couches}),0.050229,0.796186,9.482245
276,"frozenset({Couches, Sac réutilisable})",frozenset({Lingettes pour bébé}),0.050229,0.664748,9.474874
77,frozenset({Frites surgelées}),frozenset({Fromage en grains}),0.051472,0.779950,9.344296
76,frozenset({Fromage en grains}),frozenset({Frites surgelées}),0.051472,0.616667,9.344296
288,"frozenset({Tomates, Sac réutilisable})",frozenset({Laitue romaine}),0.060180,0.826342,7.604140
103,frozenset({Tomates}),frozenset({Laitue romaine}),0.066854,0.825827,7.599402


In [9]:
products.head()


,Item,Item_EN,Category,Price
0,Steak de boeuf,Beef steak,Viandes,18.99
1,Saucisses,Sausages,Viandes,7.49
2,Poitrine de poulet,Chicken breast,Viandes,12.99
3,Bacon,Bacon,Viandes,6.99
4,Saumon frais,Fresh salmon,Poissons,16.49


### Checking Product Names Against the Catalogue

Every product used in the selected rules must exist in the product catalogue. The following check identifies any missing product names before export.


In [10]:
rule_products = set()

for _, row in selected_rules.iterrows():
    rule_products.update(row["antecedents"])
    rule_products.update(row["consequents"])

catalogue_products = set(products["Item"])
missing_products = rule_products - catalogue_products

print("Unique products in selected rules:", len(rule_products))
print("Products missing from catalogue:", missing_products)


Unique products in selected rules: 50
Products missing from catalogue: set()


### Exporting Recommendation Rules

Each exported rule stores all antecedent products in a list. This preserves multi-item rules instead of silently reducing them to a single triggering product.


In [11]:
rules_export = []

for _, row in selected_rules.iterrows():
    antecedent_items = sorted(list(row["antecedents"]))
    consequent_item = next(iter(row["consequents"]))

    rules_export.append({
        "antecedents": antecedent_items,
        "consequent": consequent_item,
        "support": round(float(row["support"]), 6),
        "confidence": round(float(row["confidence"]), 6),
        "lift": round(float(row["lift"]), 6)
    })

print("Rules prepared:", len(rules_export))
print("Example exported rule:", rules_export[0] if rules_export else "No rules found")

product_lookup = products.set_index("Item").to_dict("index")

for rule in rules_export:
    rule["antecedents_info"] = [
        product_lookup[item] for item in rule["antecedents"]
    ]
    rule["consequent_info"] = product_lookup[rule["consequent"]]

print("Catalogue information added!")
print("Example with catalogue information:",
      rules_export[0] if rules_export else "No rules found")


Rules prepared: 215
Example exported rule: {'antecedents': ['Pâtes alimentaires'], 'consequent': 'Ail', 'support': 0.071027, 'confidence': 0.52866, 'lift': 6.254402}
Catalogue information added!
Example with catalogue information: {'antecedents': ['Pâtes alimentaires'], 'consequent': 'Ail', 'support': 0.071027, 'confidence': 0.52866, 'lift': 6.254402, 'antecedents_info': [{'Item_EN': 'Pasta', 'Category': 'Épicerie', 'Price': 2.49}], 'consequent_info': {'Item_EN': 'Garlic', 'Category': 'Fruits et légumes', 'Price': 1.49}}


In [12]:
import json

with open("../models/rules.json", "w", encoding="utf-8") as file:
    json.dump(rules_export, file, ensure_ascii=False, indent=4)

print("Rules successfully saved!")

with open("../models/rules.json", "r", encoding="utf-8") as file:
    saved_rules = json.load(file)

print("Saved rules:", len(saved_rules))
print("First saved rule:", saved_rules[0] if saved_rules else "No rules found")


Rules successfully saved!
Saved rules: 215
First saved rule: {'antecedents': ['Pâtes alimentaires'], 'consequent': 'Ail', 'support': 0.071027, 'confidence': 0.52866, 'lift': 6.254402, 'antecedents_info': [{'Item_EN': 'Pasta', 'Category': 'Épicerie', 'Price': 2.49}], 'consequent_info': {'Item_EN': 'Garlic', 'Category': 'Fruits et légumes', 'Price': 1.49}}


### Updating Metadata

The metadata is generated from the rules actually exported in this run. The minimum lift is recorded as 1.0, while the selection condition is strictly greater than 1.


In [13]:
recommendable_products = set(
    rule["consequent"] for rule in saved_rules
)

total_catalogue_products = len(products)
coverage = (
    len(recommendable_products) / total_catalogue_products
) * 100 if total_catalogue_products else 0

# Prepare popular products for the app's fallback recommendations.
product_lookup = products.set_index("Item").to_dict("index")
popular_item_names = basket_frequency.sort_values(ascending=False).head(5).index.tolist()
popular_products = [
    {
        "Item": item,
        "Item_EN": product_lookup[item]["Item_EN"],
        "Category": product_lookup[item]["Category"],
        "Price": float(product_lookup[item]["Price"])
    }
    for item in popular_item_names
]

# Keep each product's observed basket support so the app can explain rare items.
product_support_by_item = {
    item: round(float(count / len(basket_df)), 6)
    for item, count in basket_frequency.items()
}

metadata = {
    "project": "Boreal Marche Recommendation Engine",
    "algorithm": "Apriori",
    "library": "mlxtend",
    "min_support": 0.05,
    "min_lift": 1.0,
    "rules_count": len(saved_rules),
    "recommendable_products_count": len(recommendable_products),
    "total_catalogue_products": total_catalogue_products,
    "recommendation_coverage_percent": round(coverage, 2),
    "mining_date": pd.Timestamp.today().date().isoformat(),
    "rule_source": "models/rules.json",
    "popular_products": popular_products,
    "product_support_by_item": product_support_by_item
}

with open("../models/metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=4)

print("Metadata successfully updated:")
print(json.dumps(metadata, ensure_ascii=False, indent=4))

Metadata successfully updated:
{
    "project": "Boreal Marche Recommendation Engine",
    "algorithm": "Apriori",
    "library": "mlxtend",
    "min_support": 0.05,
    "min_lift": 1.0,
    "rules_count": 215,
    "recommendable_products_count": 50,
    "total_catalogue_products": 98,
    "recommendation_coverage_percent": 51.02,
    "mining_date": "2026-11-03",
    "rule_source": "models/rules.json",
    "popular_products": [
        {
            "Item": "Sac réutilisable",
            "Item_EN": "Reusable bag",
            "Category": "Maison",
            "Price": 1.49
        },
        {
            "Item": "Biscuits",
            "Item_EN": "Cookies",
            "Category": "Collations",
            "Price": 3.99
        },
        {
            "Item": "Noix mélangées",
            "Item_EN": "Mixed nuts",
            "Category": "Collations",
            "Price": 8.99
        },
        {
            "Item": "Beurre d'arachide",
            "Item_EN": "Peanut butter",
      

### Recommendation Coverage

Coverage measures the percentage of catalogue products that can appear as recommendations (as a consequent) in the exported rules. Coverage also helps reveal popularity bias: products that rarely occur may not be represented in the recommendation engine.


In [14]:
print("Products that can be recommended:", len(recommendable_products))
print("Total catalogue products:", total_catalogue_products)
print("Recommendation coverage:", round(coverage, 2), "%")


Products that can be recommended: 50
Total catalogue products: 98
Recommendation coverage: 51.02 %


### Streamlit Environment Check

The Streamlit application loads the saved rules. Apriori mining is performed in this notebook, not when the application starts.


In [15]:
%pip show streamlit
import streamlit as st

print("Streamlit version:", st.__version__)


Name: streamlit
Version: 1.66.1
Summary: A faster way to build and share data apps
Home-page: https://streamlit.io
Author: 
Author-email: Snowflake Inc <hello@streamlit.io>
License: 
Location: c:\Users\Nermin\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: altair, anyio, click, itsdangerous, numpy, packaging, pandas, pillow, protobuf, pyarrow, pydeck, python-multipart, requests, starlette, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Streamlit version: 1.66.1


In [16]:
selected_rules = rules[
    (rules["lift"] > 1) &
    (rules["consequents"].apply(len) == 1)
].copy()

print("Selected recommendation rules:", len(selected_rules))
print("Rules with multi-product antecedents:",
      (selected_rules["antecedents"].apply(len) > 1).sum())

Selected recommendation rules: 215
Rules with multi-product antecedents: 81


In [22]:
# Verify the famous Beer-and-Diapers association in our dataset.

beer_ids = set(
    transactions.loc[
        transactions["Item"] == "Bière", "TransactionID"
    ]
)

diaper_ids = set(
    transactions.loc[
        transactions["Item"] == "Couches", "TransactionID"
    ]
)

total_baskets = transactions["TransactionID"].nunique()
both_count = len(beer_ids & diaper_ids)

beer_count = len(beer_ids)
diaper_count = len(diaper_ids)

support = both_count / total_baskets
confidence_beer_to_diapers = both_count / beer_count
confidence_diapers_to_beer = both_count / diaper_count
lift = confidence_beer_to_diapers / (diaper_count / total_baskets)

print("BEER AND DIAPERS — DATASET VERIFICATION")
print("Total baskets:", total_baskets)
print("Beer baskets:", beer_count)
print("Diapers baskets:", diaper_count)
print("Baskets containing both:", both_count)
print(f"Support: {support:.4%}")
print(f"Confidence (Beer → Diapers): {confidence_beer_to_diapers:.4%}")
print(f"Confidence (Diapers → Beer): {confidence_diapers_to_beer:.4%}")
print(f"Lift: {lift:.4f}")

BEER AND DIAPERS — DATASET VERIFICATION
Total baskets: 1000000
Beer baskets: 150468
Diapers baskets: 83966
Baskets containing both: 30696
Support: 3.0696%
Confidence (Beer → Diapers): 20.4004%
Confidence (Diapers → Beer): 36.5577%
Lift: 2.4296


## Ethical Considerations: The Beer-and-Diapers Example

The famous Beer-and-Diapers retail example was investigated using our actual dataset.

The results showed:
- Total baskets: 1,000,000
- Baskets containing Beer: 150,468
- Baskets containing Diapers: 83,966
- Baskets containing both products: 30,696
- Support: 3.0696%
- Confidence (Beer → Diapers): 20.4004%
- Confidence (Diapers → Beer): 36.5577%
- Lift: 2.4296

The lift value of 2.4296 indicates that Beer and Diapers occur together approximately 2.43 times as often as would be expected if their purchases were independent.

This demonstrates a positive association in our dataset. However, association does not prove causation. We cannot conclude that buying Beer causes customers to buy Diapers.

**Mining threshold limitation:** Our main Apriori analysis uses a minimum support of 5%. Since this pair has a support of 3.0696%, it does not appear in the saved rules. We verified it separately using the original transaction data.

## Ethical Considerations: The Beer-and-Diapers Example

The famous Beer-and-Diapers retail example was investigated using our actual dataset.

The results showed:
- Total baskets: 1,000,000
- Baskets containing Beer: 150,468
- Baskets containing Diapers: 83,966
- Baskets containing both products: 30,696
- Support: 3.0696%
- Confidence (Beer → Diapers): 20.4004%
- Confidence (Diapers → Beer): 36.5577%
- Lift: 2.4296

The lift value of 2.4296 indicates that Beer and Diapers occur together approximately 2.43 times as often as would be expected if their purchases were independent.

This demonstrates a positive association in our dataset. However, association does not prove causation. We cannot conclude that buying Beer causes customers to buy Diapers.

**Mining threshold limitation:** Our main Apriori analysis uses a minimum support of 5%. Since this pair has a support of 3.0696%, it does not appear in the saved rules. We verified it separately using the original transaction data.